# VibeCheck — CLIP + LoRA Aesthetic Classifier

Trains a LoRA-fine-tuned CLIP-ViT-B/32 classifier on 8 aesthetic classes and evaluates it against three baselines:
- Groq / Llama-4 Scout (production baseline, loaded from `data/eval/groq_predictions.json`)
- CLIP zero-shot
- CLIP linear probe
- **CLIP + LoRA (ours)**

**Inputs:**  `data/eval_processed/<aesthetic>/*.jpg`  
**Outputs:** `models/clip_lora_best.pt`, confusion matrices, ablation tables

## 1. Imports & config

In [ ]:
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    top_k_accuracy_score,
)
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from transformers import CLIPModel, CLIPProcessor

# ── Config ────────────────────────────────────────────────────────────────────
DATA_ROOT   = Path('data/eval_processed')   # change to 'data/eval' if no preprocessing step
MODEL_DIR   = Path('models')
GROQ_CACHE  = Path('data/eval/groq_predictions.json')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

AESTHETICS = [
    'minimalist', 'dark_academia', 'cottagecore', 'grunge',
    'coastal_grandmother', 'scandinavian', 'mid_century_modern', 'japandi', 
]
LABEL2IDX = {a: i for i, a in enumerate(AESTHETICS)}
IDX2LABEL = {i: a for a, i in LABEL2IDX.items()}

DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
SEED   = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f'Device: {DEVICE}')
print(f'Aesthetics: {AESTHETICS}')

## 2. Dataset inventory

In [ ]:
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}

records = []  # (path, label_str, label_idx)
for aesthetic in AESTHETICS:
    cls_dir = DATA_ROOT / aesthetic
    if not cls_dir.is_dir():
        print(f'  MISSING: {cls_dir}')
        continue
    imgs = [p for p in cls_dir.iterdir() if p.suffix.lower() in IMG_EXTS]
    for p in imgs:
        records.append((str(p), aesthetic, LABEL2IDX[aesthetic]))

paths  = [r[0] for r in records]
labels = [r[1] for r in records]
y      = np.array([r[2] for r in records])

from collections import Counter
counts = Counter(labels)

fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(counts.keys(), counts.values(), color='steelblue')
ax.set_title('Images per aesthetic class')
ax.set_ylabel('count')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print(f'Total: {len(records)} images across {len(counts)} classes')
print(dict(counts))

## 3. Train / val / test split  (60 / 20 / 20, stratified)

In [ ]:
sss_test = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
trainval_idx, test_idx = next(sss_test.split(paths, y))

sss_val = StratifiedShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)  # 0.25 of 0.8 = 0.20
train_idx, val_idx = next(sss_val.split([paths[i] for i in trainval_idx],
                                         y[trainval_idx]))
train_idx = trainval_idx[train_idx]
val_idx   = trainval_idx[val_idx]

print(f'Train: {len(train_idx)}  |  Val: {len(val_idx)}  |  Test: {len(test_idx)}')
print(f'Test class distribution: {Counter(y[test_idx].tolist())}')

## 4. CLIP model + LoRA injection

In [ ]:
CLIP_MODEL_NAME = 'openai/clip-vit-base-patch32'

clip_model    = CLIPModel.from_pretrained(CLIP_MODEL_NAME)
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)

# Freeze everything
for p in clip_model.parameters():
    p.requires_grad_(False)


class LoRALinear(nn.Module):
    """Low-rank adapter injected into a frozen nn.Linear."""
    def __init__(self, linear: nn.Linear, r: int = 8):
        super().__init__()
        self.linear = linear
        d_in, d_out = linear.in_features, linear.out_features
        self.A = nn.Parameter(torch.randn(d_in, r) * 0.01)
        self.B = nn.Parameter(torch.zeros(r, d_out))
        self.scale = 1.0 / r

    def forward(self, x):
        return self.linear(x) + (x @ self.A @ self.B) * self.scale


def inject_lora(model: CLIPModel, r: int = 8) -> CLIPModel:
    """Replace q/k/v/out projections in the vision encoder with LoRA wrappers."""
    for layer in model.vision_model.encoder.layers:
        attn = layer.self_attn
        for proj in ('q_proj', 'k_proj', 'v_proj', 'out_proj'):
            setattr(attn, proj, LoRALinear(getattr(attn, proj), r=r))
    return model


LORA_RANK = 8
clip_model = inject_lora(clip_model, r=LORA_RANK).to(DEVICE)

trainable = sum(p.numel() for p in clip_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in clip_model.parameters())
print(f'Trainable params: {trainable:,}  ({100*trainable/total:.2f}% of {total:,})')

## 5. Dataset & DataLoader

In [ ]:
AUGMENT = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, saturation=0.2, hue=0.05),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
])


class VibeDataset(Dataset):
    def __init__(self, indices, augment=False):
        self.indices = indices
        self.augment  = augment

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx   = self.indices[i]
        path  = paths[idx]
        label = int(y[idx])
        img   = Image.open(path).convert('RGB')
        if self.augment:
            img = AUGMENT(img)
        inputs = clip_processor(images=img, return_tensors='pt')
        return inputs['pixel_values'].squeeze(0), label


BATCH = 16
train_loader = DataLoader(VibeDataset(train_idx, augment=True),  batch_size=BATCH, shuffle=True,  num_workers=0)
val_loader   = DataLoader(VibeDataset(val_idx,   augment=False), batch_size=BATCH, shuffle=False, num_workers=0)
test_loader  = DataLoader(VibeDataset(test_idx,  augment=False), batch_size=BATCH, shuffle=False, num_workers=0)

print(f'Batches — train: {len(train_loader)}  val: {len(val_loader)}  test: {len(test_loader)}')

## 6. Classifier head + optimizer

In [ ]:
head = nn.Linear(512, len(AESTHETICS)).to(DEVICE)

lora_params = [
    p for m in clip_model.modules()
    if isinstance(m, LoRALinear)
    for p in [m.A, m.B]
]

optimizer = torch.optim.AdamW([
    {'params': lora_params,        'lr': 1e-4},
    {'params': head.parameters(),  'lr': 1e-3},
], weight_decay=1e-4)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

print('Head:', head)
print('Optimizer param groups:', [{'lr': g['lr'], 'n': len(g['params'])} for g in optimizer.param_groups])

## 7. Training loop

In [ ]:
EPOCHS        = 30
PATIENCE      = 7
SAVE_PATH     = MODEL_DIR / 'clip_lora_best.pt'

best_val_acc      = 0.0
patience_counter  = 0
train_losses, val_accs = [], []

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    clip_model.train(); head.train()
    epoch_loss = 0.0
    for pixels, label_ids in train_loader:
        pixels, label_ids = pixels.to(DEVICE), label_ids.to(DEVICE)
        feats = clip_model.get_image_features(pixel_values=pixels)
        feats = feats / feats.norm(dim=-1, keepdim=True)
        loss  = criterion(head(feats), label_ids)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    # ── Validate ──
    clip_model.eval(); head.eval()
    correct = total = 0
    with torch.no_grad():
        for pixels, label_ids in val_loader:
            pixels, label_ids = pixels.to(DEVICE), label_ids.to(DEVICE)
            feats = clip_model.get_image_features(pixel_values=pixels)
            feats = feats / feats.norm(dim=-1, keepdim=True)
            preds = head(feats).argmax(dim=-1)
            correct += (preds == label_ids).sum().item()
            total   += len(label_ids)

    val_acc   = correct / total
    avg_loss  = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    val_accs.append(val_acc)

    print(f'Epoch {epoch:>3}/{EPOCHS}  loss={avg_loss:.4f}  val_acc={val_acc:.3f}', end='')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save({'model': clip_model.state_dict(), 'head': head.state_dict()}, SAVE_PATH)
        print('  ✓ saved')
    else:
        patience_counter += 1
        print(f'  (patience {patience_counter}/{PATIENCE})')
        if patience_counter >= PATIENCE:
            print('Early stopping.')
            break

print(f'\nBest val accuracy: {best_val_acc:.3f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses); ax1.set_title('Training loss'); ax1.set_xlabel('epoch')
ax2.plot(val_accs);     ax2.set_title('Val accuracy');  ax2.set_xlabel('epoch')
plt.tight_layout(); plt.show()

## 8. Test-set evaluation — all four systems

### 8a. CLIP + LoRA (load best checkpoint)

In [ ]:
checkpoint = torch.load(SAVE_PATH, map_location=DEVICE)
clip_model.load_state_dict(checkpoint['model'])
head.load_state_dict(checkpoint['head'])
clip_model.eval(); head.eval()

lora_logits_all, lora_true = [], []
with torch.no_grad():
    for pixels, label_ids in test_loader:
        pixels = pixels.to(DEVICE)
        feats  = clip_model.get_image_features(pixel_values=pixels)
        feats  = feats / feats.norm(dim=-1, keepdim=True)
        lora_logits_all.append(head(feats).cpu())
        lora_true.extend(label_ids.tolist())

lora_logits = torch.cat(lora_logits_all).numpy()
lora_preds  = lora_logits.argmax(axis=1)
lora_true   = np.array(lora_true)

lora_top1 = accuracy_score(lora_true, lora_preds)
lora_top3 = top_k_accuracy_score(lora_true, lora_logits, k=3)
print(f'CLIP + LoRA   top-1={lora_top1:.3f}  top-3={lora_top3:.3f}')

### 8b. CLIP zero-shot

In [ ]:
# Prompt engineering: each class gets a short descriptive sentence
CLASS_PROMPTS = {
    'minimalist':          'a minimalist room or outfit with clean lines and neutral tones',
    'dark_academia':       'a dark academia aesthetic with books, warm candlelight, and rich earth tones',
    'cottagecore':         'a cottagecore aesthetic with floral patterns, linen, and pastoral nature',
    'grunge':              'a grunge aesthetic with distressed clothing, dark colors, and edgy style',
    'coastal_grandmother': 'a coastal grandmother aesthetic with linen, beige, and relaxed seaside style',
    'scandinavian':        'a scandinavian interior with light wood, white walls, and functional design',
    'mid_century_modern':  'a mid-century modern interior with organic shapes and retro warm tones',
    'japandi':             'a japandi aesthetic blending Japanese minimalism and Scandinavian warmth',
}

prompts = [CLASS_PROMPTS[a] for a in AESTHETICS]
text_inputs = clip_processor(text=prompts, return_tensors='pt', padding=True).to(DEVICE)

with torch.no_grad():
    # Use frozen CLIP text encoder for zero-shot (no LoRA on text side)
    base_clip = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE)
    text_feats = base_clip.get_text_features(**text_inputs)
    text_feats = text_feats / text_feats.norm(dim=-1, keepdim=True)

zs_logits_all, zs_true = [], []
base_clip.eval()
with torch.no_grad():
    for pixels, label_ids in test_loader:
        pixels = pixels.to(DEVICE)
        img_feats = base_clip.get_image_features(pixel_values=pixels)
        img_feats = img_feats / img_feats.norm(dim=-1, keepdim=True)
        sims = img_feats @ text_feats.T          # cosine similarity
        zs_logits_all.append(sims.cpu())
        zs_true.extend(label_ids.tolist())

zs_logits = torch.cat(zs_logits_all).numpy()
zs_preds  = zs_logits.argmax(axis=1)
zs_true   = np.array(zs_true)

zs_top1 = accuracy_score(zs_true, zs_preds)
zs_top3 = top_k_accuracy_score(zs_true, zs_logits, k=3)
print(f'CLIP zero-shot  top-1={zs_top1:.3f}  top-3={zs_top3:.3f}')

### 8c. CLIP linear probe

In [ ]:
# Extract frozen CLIP embeddings for all splits
def extract_embeddings(loader, model):
    feats_list, labels_list = [], []
    model.eval()
    with torch.no_grad():
        for pixels, label_ids in loader:
            pixels = pixels.to(DEVICE)
            f = model.get_image_features(pixel_values=pixels)
            f = f / f.norm(dim=-1, keepdim=True)
            feats_list.append(f.cpu().numpy())
            labels_list.extend(label_ids.tolist())
    return np.vstack(feats_list), np.array(labels_list)

X_train_emb, y_train_emb = extract_embeddings(train_loader, base_clip)
X_val_emb,   y_val_emb   = extract_embeddings(val_loader,   base_clip)
X_test_emb,  y_test_emb  = extract_embeddings(test_loader,  base_clip)

# Sweep C on val set
best_C, best_probe_acc = 0.1, 0.0
for C in [0.01, 0.1, 1.0, 10.0]:
    clf = LogisticRegression(C=C, max_iter=1000, random_state=SEED)
    clf.fit(X_train_emb, y_train_emb)
    acc = accuracy_score(y_val_emb, clf.predict(X_val_emb))
    print(f'  C={C:<6}  val_acc={acc:.3f}')
    if acc > best_probe_acc:
        best_probe_acc, best_C = acc, C

probe = LogisticRegression(C=best_C, max_iter=1000, random_state=SEED)
probe.fit(np.vstack([X_train_emb, X_val_emb]),
          np.concatenate([y_train_emb, y_val_emb]))

probe_logits = probe.predict_proba(X_test_emb)
probe_preds  = probe_logits.argmax(axis=1)
probe_top1   = accuracy_score(y_test_emb, probe_preds)
probe_top3   = top_k_accuracy_score(y_test_emb, probe_logits, k=3)
print(f'\nCLIP linear probe (C={best_C})  top-1={probe_top1:.3f}  top-3={probe_top3:.3f}')

### 8d. Groq baseline (from cache)

In [ ]:
groq_top1 = groq_top3 = None

if GROQ_CACHE.exists():
    groq_cache = json.loads(GROQ_CACHE.read_text())
    eval_root  = Path('data/eval')

    groq_y_true, groq_y_pred = [], []
    for idx in test_idx:
        abs_path = Path(paths[idx])
        # Try relative key (new format) then absolute (old format)
        try:
            rel_key = abs_path.relative_to(eval_root.resolve()).as_posix()
        except ValueError:
            rel_key = str(abs_path)
        pred = groq_cache.get(rel_key) or groq_cache.get(str(abs_path))
        if pred is not None and pred in LABEL2IDX:
            groq_y_true.append(y[idx])
            groq_y_pred.append(LABEL2IDX[pred])

    if groq_y_true:
        groq_top1 = accuracy_score(groq_y_true, groq_y_pred)
        print(f'Groq / Llama-4 Scout  top-1={groq_top1:.3f}  '
              f'(n={len(groq_y_true)}/{len(test_idx)} test images cached)')
    else:
        print('No Groq predictions matched test images.')
else:
    print(f'Groq cache not found at {GROQ_CACHE}.')
    print('Run: GROQ_API_KEY=... python scripts/export_groq_baseline.py')

## 9. Summary table

In [ ]:
import pandas as pd

rows = [
    {'System': 'Groq / Llama-4 Scout',  'Top-1': groq_top1,   'Top-3': '—',        'Params': '0 (frozen API)',  'Latency': '~2 s/req'},
    {'System': 'CLIP zero-shot',         'Top-1': zs_top1,     'Top-3': zs_top3,     'Params': '0 (frozen)',      'Latency': '~30 ms'},
    {'System': 'CLIP linear probe',      'Top-1': probe_top1,  'Top-3': probe_top3,  'Params': '~4 K',            'Latency': '~30 ms'},
    {'System': 'CLIP + LoRA (ours)',     'Top-1': lora_top1,   'Top-3': lora_top3,   'Params': '~0.3 M (0.3%)',   'Latency': '~30 ms'},
]

def fmt(v):
    if v is None or v == '—': return '—'
    return f'{float(v):.3f}'

df = pd.DataFrame(rows)
df['Top-1'] = df['Top-1'].apply(fmt)
df['Top-3'] = df['Top-3'].apply(fmt)
df.set_index('System', inplace=True)
print(df.to_string())
df

## 10. Confusion matrices

In [ ]:
SHORT = [a.replace('_', '\n') for a in AESTHETICS]

def plot_cm(y_true, y_pred, title, ax):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(AESTHETICS))), normalize='true')
    sns.heatmap(cm, annot=True, fmt='.2f', xticklabels=SHORT, yticklabels=SHORT,
                cmap='Blues', vmin=0, vmax=1, ax=ax, linewidths=0.3)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
plot_cm(zs_true,     zs_preds,    'CLIP zero-shot',      axes[0])
plot_cm(y_test_emb,  probe_preds, 'CLIP linear probe',   axes[1])
plot_cm(lora_true,   lora_preds,  'CLIP + LoRA (ours)',  axes[2])
plt.suptitle('Confusion matrices (row-normalised) — held-out test set', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('models/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to models/confusion_matrices.png')

## 11. Per-class accuracy (LoRA)

In [ ]:
per_class = {}
for cls_idx, cls_name in IDX2LABEL.items():
    mask = lora_true == cls_idx
    if mask.sum() == 0:
        per_class[cls_name] = None
        continue
    per_class[cls_name] = accuracy_score(lora_true[mask], lora_preds[mask])

names = list(per_class.keys())
accs  = [v if v is not None else 0 for v in per_class.values()]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(names, accs, color=['#e74c3c' if a < 0.5 else '#2ecc71' for a in accs])
ax.axhline(lora_top1, color='navy', linestyle='--', label=f'macro avg={lora_top1:.3f}')
ax.set_ylim(0, 1.05)
ax.set_ylabel('accuracy')
ax.set_title('CLIP + LoRA — per-class accuracy on test set')
ax.legend()
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('models/per_class_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

for name, acc in sorted(per_class.items(), key=lambda x: x[1] or 0):
    print(f'  {name:<24} {acc:.3f}' if acc is not None else f'  {name:<24} —')

## 12. LoRA rank ablation

In [ ]:
# Sweep r ∈ {4, 8, 16, 32} — trains a fresh model for each rank
# Skip this cell if time is limited; r=8 is the main result.

RANK_SWEEP = [4, 8, 16, 32]
rank_results = {}

def quick_train(r, epochs=15):
    m = CLIPModel.from_pretrained(CLIP_MODEL_NAME)
    for p in m.parameters(): p.requires_grad_(False)
    m = inject_lora(m, r=r).to(DEVICE)
    h = nn.Linear(512, len(AESTHETICS)).to(DEVICE)
    lp = [p for mod in m.modules() if isinstance(mod, LoRALinear) for p in [mod.A, mod.B]]
    opt = torch.optim.AdamW([{'params': lp, 'lr': 1e-4}, {'params': h.parameters(), 'lr': 1e-3}])
    crit = nn.CrossEntropyLoss(label_smoothing=0.05)
    for _ in range(epochs):
        m.train(); h.train()
        for pixels, labs in train_loader:
            pixels, labs = pixels.to(DEVICE), labs.to(DEVICE)
            f = m.get_image_features(pixel_values=pixels)
            f = f / f.norm(dim=-1, keepdim=True)
            loss = crit(h(f), labs)
            opt.zero_grad(); loss.backward(); opt.step()
    # Eval on test
    m.eval(); h.eval(); all_logits, all_true = [], []
    with torch.no_grad():
        for pixels, labs in test_loader:
            f = m.get_image_features(pixel_values=pixels.to(DEVICE))
            f = f / f.norm(dim=-1, keepdim=True)
            all_logits.append(h(f).cpu().numpy())
            all_true.extend(labs.tolist())
    logits = np.vstack(all_logits)
    return accuracy_score(all_true, logits.argmax(axis=1))

for r in RANK_SWEEP:
    acc = quick_train(r)
    rank_results[r] = acc
    print(f'  r={r:<4}  test_top1={acc:.3f}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(rank_results.keys()), list(rank_results.values()), marker='o')
ax.set_xlabel('LoRA rank (r)'); ax.set_ylabel('test top-1')
ax.set_title('LoRA rank ablation')
ax.set_xticks(RANK_SWEEP)
plt.tight_layout()
plt.savefig('models/rank_ablation.png', dpi=150)
plt.show()

## 13. Data scale curve (accuracy vs. training set size)

In [ ]:
# Train on 25%, 50%, 75%, 100% of training data — tests whether more data helps
FRACS = [0.25, 0.50, 0.75, 1.0]
scale_results = {}

for frac in FRACS:
    n = max(len(AESTHETICS), int(len(train_idx) * frac))   # at least 1/class
    sub_idx = train_idx[:n]
    sub_loader = DataLoader(VibeDataset(sub_idx, augment=True), batch_size=BATCH, shuffle=True, num_workers=0)

    m = CLIPModel.from_pretrained(CLIP_MODEL_NAME)
    for p in m.parameters(): p.requires_grad_(False)
    m = inject_lora(m, r=8).to(DEVICE)
    h = nn.Linear(512, len(AESTHETICS)).to(DEVICE)
    lp = [p for mod in m.modules() if isinstance(mod, LoRALinear) for p in [mod.A, mod.B]]
    opt = torch.optim.AdamW([{'params': lp, 'lr': 1e-4}, {'params': h.parameters(), 'lr': 1e-3}])
    crit = nn.CrossEntropyLoss(label_smoothing=0.05)

    for _ in range(20):
        m.train(); h.train()
        for pixels, labs in sub_loader:
            f = m.get_image_features(pixel_values=pixels.to(DEVICE))
            f = f / f.norm(dim=-1, keepdim=True)
            loss = crit(h(f), labs.to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step()

    m.eval(); h.eval(); all_logits, all_true = [], []
    with torch.no_grad():
        for pixels, labs in test_loader:
            f = m.get_image_features(pixel_values=pixels.to(DEVICE))
            f = f / f.norm(dim=-1, keepdim=True)
            all_logits.append(h(f).cpu().numpy())
            all_true.extend(labs.tolist())
    acc = accuracy_score(all_true, np.vstack(all_logits).argmax(axis=1))
    scale_results[n] = acc
    print(f'  n={n:<5} ({int(frac*100):>3}%)  test_top1={acc:.3f}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(scale_results.keys()), list(scale_results.values()), marker='o')
ax.set_xlabel('Training images'); ax.set_ylabel('test top-1')
ax.set_title('Data scale curve')
plt.tight_layout()
plt.savefig('models/data_scale_curve.png', dpi=150)
plt.show()

## 14. Failure mode analysis

In [ ]:
# Show the 12 most confidently wrong predictions
softmax = torch.softmax(torch.tensor(lora_logits), dim=-1).numpy()
wrong_mask   = lora_preds != lora_true
wrong_indices = np.where(wrong_mask)[0]
wrong_conf   = softmax[wrong_indices, lora_preds[wrong_indices]]
top_wrong    = wrong_indices[np.argsort(-wrong_conf)][:12]

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, wi in zip(axes.flat, top_wrong):
    test_sample_idx = test_idx[wi]
    img = Image.open(paths[test_sample_idx]).convert('RGB')
    ax.imshow(img)
    true_name = IDX2LABEL[lora_true[wi]]
    pred_name = IDX2LABEL[lora_preds[wi]]
    conf = wrong_conf[np.where(top_wrong == wi)[0][0]]
    ax.set_title(f'True: {true_name}\nPred: {pred_name} ({conf:.2f})', fontsize=8)
    ax.axis('off')

plt.suptitle('Most confident errors — CLIP + LoRA', fontsize=12)
plt.tight_layout()
plt.savefig('models/failure_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

## 15. Export results for report

In [ ]:
results = {
    'groq':         {'top1': groq_top1,   'top3': None},
    'clip_zeroshot':{'top1': float(zs_top1),     'top3': float(zs_top3)},
    'clip_probe':   {'top1': float(probe_top1),  'top3': float(probe_top3)},
    'clip_lora':    {'top1': float(lora_top1),   'top3': float(lora_top3)},
    'rank_ablation': {str(k): float(v) for k, v in rank_results.items()},
    'scale_curve':   {str(k): float(v) for k, v in scale_results.items()},
    'per_class_lora': {k: float(v) if v is not None else None for k, v in per_class.items()},
    'n_test': int(len(test_idx)),
    'n_train': int(len(train_idx)),
    'lora_rank': LORA_RANK,
}

out_path = Path('models/eval_results.json')
out_path.write_text(json.dumps(results, indent=2))
print(f'Results saved to {out_path}')
print(json.dumps(results, indent=2))